This notebook performs data cleaning, merging, and feature engineering on the Olist e-commerce dataset. The goal is to create a unified master dataset for downstream revenue analysis and customer modeling.

In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

In [4]:
Orders= pd.read_csv(r'/Users/tendaisinkala/Documents/Data Sets/olist_orders_dataset.csv')
Customers= pd.read_csv(r'/Users/tendaisinkala/Documents/Data Sets/olist_customers_dataset.csv')
Orders_items= pd.read_csv(r'/Users/tendaisinkala/Documents/Data Sets/olist_order_items_dataset.csv')
Payments= pd.read_csv(r'/Users/tendaisinkala/Documents/Data Sets/olist_order_payments_dataset.csv')
Products= pd.read_csv(r'/Users/tendaisinkala/Documents/Data Sets/olist_products_dataset.csv')
Reviews= pd.read_csv(r'/Users/tendaisinkala/Documents/Data Sets/olist_order_reviews_dataset.csv')

In [5]:
Orders.head()
Orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [6]:
Customers.head()
Customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [7]:
Orders_items.head()
Orders_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [8]:
Payments.head()
Payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


In [9]:
Products.head()
Products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [10]:
Reviews.head()
Reviews.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [12]:
Orders['order_purcahse_timestamp']= pd.to_datetime(Orders['order_purchase_timestamp'])
Orders['order_delvered_customer_date']= pd.to_datetime(Orders['order_delivered_customer_date'])
Orders['order_estimated_delivery_date']= pd.to_datetime(Orders['order_estimated_delivery_date'])


In [14]:
Order_data= Orders_items.merge(Payments, on='order_id', how='left')

In [16]:
Order_data= Order_data.merge(Products, on='product_id', how='left')

In [17]:
Order_data= Order_data.merge(Orders, on='order_id', how='left')

In [18]:
Master_df= Order_data.merge(Customers, on='customer_id', how='left')


In [20]:
Master_df['total_item_value']= Master_df['price'] + Master_df['freight_value']

In [21]:
Order_totals= Master_df.groupby('order_id')['total_item_value'].sum().reset_index()
Order_totals.rename(columns={'total_item_value':'total_order_value'}, inplace=True)

Master_df= Master_df.merge(Order_totals,  on='order_id', how='left')

In [24]:
Master_df['order_purchase_timestamp']= pd.to_datetime(
    Master_df['order_purchase_timestamp'],
    errors='coerce'
)

Master_df['order_month']= Master_df['order_purchase_timestamp'].dt.to_period('M')
Master_df['order_year']= Master_df['order_purcahse_timestamp'].dt.year

In [28]:
Master_df['order_delivered_customer_date']= pd.to_datetime(
    Master_df['order_delivered_customer_date'],
    errors='coerce'
)

Master_df['delivery_time_days']= (
    Master_df['order_delivered_customer_date'] -
    Master_df['order_purchase_timestamp']
).dt.days


In [29]:
Master_df['late_delivery_flag']= np.where(
    Master_df['order_delivered_customer_date']> Master_df['order_estimated_delivery_date'],
    1,0
)

In [30]:
Master_df.isnull().sum().sort_values(ascending=False)

delivery_time_days               2567
order_delivered_customer_date    2567
order_delvered_customer_date     2567
product_description_lenght       1698
product_category_name            1698
product_photos_qty               1698
product_name_lenght              1698
order_delivered_carrier_date     1245
product_height_cm                  20
product_length_cm                  20
product_weight_g                   20
product_width_cm                   20
order_approved_at                  15
payment_value                       3
payment_type                        3
payment_sequential                  3
payment_installments                3
customer_city                       0
customer_zip_code_prefix            0
order_id                            0
customer_state                      0
total_item_value                    0
total_order_value                   0
order_month                         0
order_year                          0
customer_unique_id                  0
customer_id 

In [33]:
missing_pct=(Master_df.isnull().sum()/len(Master_df))* 100
missing_pct.sort_values(ascending=False)

product_description_lenght       1.415185
product_category_name            1.415185
product_photos_qty               1.415185
product_name_lenght              1.415185
product_height_cm                0.017386
product_length_cm                0.017386
product_weight_g                 0.017386
product_width_cm                 0.017386
order_approved_at                0.013039
order_delvered_customer_date     0.006954
order_delivered_customer_date    0.006954
delivery_time_days               0.006954
payment_type                     0.002608
payment_value                    0.002608
payment_sequential               0.002608
payment_installments             0.002608
order_delivered_carrier_date     0.001739
customer_city                    0.000000
customer_zip_code_prefix         0.000000
order_id                         0.000000
customer_state                   0.000000
total_item_value                 0.000000
total_order_value                0.000000
order_month                      0

In [32]:
Master_df= Master_df[Master_df['order_status']== 'delivered']

In [34]:
Master_df= Master_df.dropna(subset=['order_delivered_customer_date'])

In [35]:
Master_df['product_category_name']= Master_df['product_category_name'].fillna('unknown')

In [36]:
Master_df= Master_df.dropna(subset=['price'])

In [37]:
Master_df= Master_df.dropna(subset=['delivery_time_days'])

In [38]:
Master_df.isnull().sum()

order_id                            0
order_item_id                       0
product_id                          0
seller_id                           0
shipping_limit_date                 0
price                               0
freight_value                       0
payment_sequential                  3
payment_type                        3
payment_installments                3
payment_value                       3
product_category_name               0
product_name_lenght              1628
product_description_lenght       1628
product_photos_qty               1628
product_weight_g                   20
product_length_cm                  20
product_height_cm                  20
product_width_cm                   20
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  15
order_delivered_carrier_date        1
order_delivered_customer_date       0
order_estimated_delivery_date       0
order_purcah

In [39]:
missing_pct.sort_values(ascending=False)

product_description_lenght       1.415185
product_category_name            1.415185
product_photos_qty               1.415185
product_name_lenght              1.415185
product_height_cm                0.017386
product_length_cm                0.017386
product_weight_g                 0.017386
product_width_cm                 0.017386
order_approved_at                0.013039
order_delvered_customer_date     0.006954
order_delivered_customer_date    0.006954
delivery_time_days               0.006954
payment_type                     0.002608
payment_value                    0.002608
payment_sequential               0.002608
payment_installments             0.002608
order_delivered_carrier_date     0.001739
customer_city                    0.000000
customer_zip_code_prefix         0.000000
order_id                         0.000000
customer_state                   0.000000
total_item_value                 0.000000
total_order_value                0.000000
order_month                      0

In [40]:
Master_df['product_category_name']= Master_df['product_category_name'].fillna('unknown')

In [41]:
Master_df= Master_df.dropna(subset=[
    'product_description_lenght',
    'product_name_lenght',
    'product_photos_qty'
])

In [42]:
Master_df= Master_df.dropna(subset=['product_height_cm'])

In [43]:
Master_df.isnull().sum().sort_values(ascending=False)

order_approved_at                14
payment_sequential                3
payment_installments              3
payment_type                      3
payment_value                     3
order_delivered_carrier_date      1
total_order_value                 0
total_item_value                  0
order_month                       0
order_year                        0
delivery_time_days                0
order_purchase_timestamp          0
customer_state                    0
customer_city                     0
customer_zip_code_prefix          0
customer_unique_id                0
order_delvered_customer_date      0
order_purcahse_timestamp          0
order_estimated_delivery_date     0
order_delivered_customer_date     0
order_id                          0
customer_id                       0
order_status                      0
order_item_id                     0
product_width_cm                  0
product_height_cm                 0
product_length_cm                 0
product_weight_g            

In [48]:
Master_df.to_csv('master_cleaned_dataset.csv', index=False)